### Week 2 Day 4: Airline AI Assistant 

In this project, we are building **FlightAI**, an AI Customer Support assistant for an airline. We will build this in three phases to understand how Large Language Models (LLMs) can interact with real-world tools and databases.

### Phase 1: The Basic Chatbot
Before adding complex tools, we must first establish a baseline. We need to connect to our local LLM and give it a "Persona" using a system prompt so it knows it works for an airline. 

In this phase, the bot can chat politely, but it has no access to real ticket prices. If you ask it for a price, it will either guess or tell you it doesn't know.

In [1]:
import json
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv


# Automatically find the .env file in parent directories and load it
load_dotenv(find_dotenv(), override=True)

# 1. Connect to our local Ollama server
client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
) # No API key is required for local Ollama server

MODEL = "gemini-2.5-flash"  # You can change this to any other model you have available

# 2. Define the Airline Personalization Prompt
system_prompt = """
You are a helpful customer support assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [3]:
# 3. Define the basic chat function
def chat(message, history):
    # Set up our System Instruction
    messages = [{"role": "system", "content": system_prompt}]

    # just append the whole history list directly.
    messages.extend(history)

    # Append the current user message at the end
    messages.append({"role": "user", "content": message})


    # Call the model
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content

In [4]:
# 4. Launch the Phase 1 UI
gr.ChatInterface(
    fn=chat,
    title="FlightAI Assistant (Phase 1)"
).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


### Tools
* Tools are an incredibly powerful feature provided by the frontier LLMs.
* With tools, you can write a function, and have the LLM call that function as part of its response.

## Phase 2: Single Tool Calling (Dummy Data)
Before handling complex scenarios, we need to understand the absolute basics of how a tool works. We will create a Python function with dummy dictionary data and describe it to the AI. 

We will use a simple `if` statement to check if the AI wants to use a tool. If it does, we will only grab the **first** tool call (`tool_calls[0]`), run our Python function, and send the single result back to the AI.

In [5]:
# 1. Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [6]:
# 2. Describe the tool to the AI using JSON
price_tool = {
    "type": "function",
    "function": {
        "name": "get_ticket_price",
        "description": "Get the price of a return ticket to a destination city.",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string",
                    "description": "The city that the customer wants to travel to"
                }
            },
            "required": ["destination_city"],
            "additionalProperties": False
        }
    }
}


In [7]:
tools=[price_tool]

In [8]:
# 3. Dedicated Helper Function: Process the Tool Call

def handle_tool_call(message):
    # we only grab the very first tool call in this phase
    tool_call = message.tool_calls[0]

    if tool_call.function.name == "get_ticket_price":
        # Parse the JSON argument
        argument = json.loads(tool_call.function.arguments)
        city = argument.get('destination_city')

        # Run the Python function
        price_details = get_ticket_price(city)

        # assign the formatted tool result
        response = {
            "role" : "tool",
            "content" : price_details,
            "tool_call_id":tool_call.id
        }

        return response


In [9]:
# 4. Main Chat Function
def chat(message, history):
    # Set up our System Instruction
    messages = [{"role": "system", "content": system_prompt}]

    # just append the whole history list directly.
    messages.extend(history)

    # Append the current user message at the end
    messages.append({"role": "user", "content": message})


    # Call the model
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    # Use a simple If statement to check if the AI requested a tool
    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message

        # Pass the message to our dedicated helper function
        tool_result = handle_tool_call(message_obj)

        # Append the AI's request and our result
        messages.append(message_obj)
        messages.append(tool_result)

        # Send back to the AI for the dinal text answer
        response = client.chat.completions.create(
            model= MODEL,
            messages=messages
        )

    # Return the finsl text 
    return response.choices[0].message.content


In [10]:
gr.ChatInterface(fn = chat,title = "FlightAI Assistant").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


### Phase 3: Advanced Tool Calling (Parallel & Sequential)
Our Phase 2 code is too basic. It fails if the AI tries to run two tools at once, or if it needs to chain tools together.

### The Upgrades:
1. **`handle_tool_calls` function (Parallel execution):** Instead of grabbing only `tool_calls[0]`, we use a `for` loop to process *all* tools the AI asked for simultaneously. This allows the AI to answer things like *"What is the price for London AND Paris?"*
2. **`while` loop (Sequential execution):** We replaced the `if` statement with a `while` loop. Now, the AI can call a tool, look at the result, and if it realizes it needs more data, it will loop around and call another tool before finally answering the user!

In [11]:
# 1. New helper function to handle multiple tools at the same time
def handle_tool_calls(message):
    responses = []
    # Loop through EVERY tool call the AI requested
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            
            # Run the Python function
            price_details = get_ticket_price(city)
            
            # Add the result to our list
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [12]:
def chat(message, history):
    # Set up our System Instruction
    messages = [{"role": "system", "content": system_prompt}]

    # just append the whole history list directly.
    messages.extend(history)

    # Append the current user message at the end
    messages.append({"role": "user", "content": message})

    # Initial call to the LLM
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    # 3. Keep looping as long as the AI wants to use tools!

    while response.choices[0].finish_reason == "tool_calls":

        message_obj = response.choices[0].message

        # Process ALL requested tool calls using our new helper function
        tool_result = handle_tool_calls(message_obj)

        # Append the AI's request and our results to the message history
        messages.append(message_obj.model_dump(exclude_none=True))
        messages.extend(tool_result)

        # Send back to the AI. Notice we pass tools=tools again so it can keep calling them!
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    # Return the final text
    return response.choices[0].message.content

In [13]:
gr.ChatInterface(fn=chat, title="FlightAI Assistant").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## Phase 4: Connecting the Tool to a Real Database (SQLite)
In a real-world scenario, FlightAI would have thousands of flights, and prices change constantly. Hardcoding them in a Python dictionary like we did in Phase 2 & 3 is impossible.

We need to upgrade our Python Tool to connect to a real **SQLite Database**. When the AI asks for a price, our Python tool will run a SQL `SELECT` query, fetch the live data, and hand it back to the AI.

In [14]:
import sqlite3

# 1. Create a local SQLite database file named 'flights.db'
conn = sqlite3.connect("flights.db", check_same_thread=False)
cursor = conn.cursor()

# 2. Set up our table and insert some data
cursor.execute("CREATE TABLE IF NOT EXISTS tickets (city TEXT, price TEXT)")
cursor.execute("DELETE FROM tickets") # Clear old data for our test
cursor.execute("INSERT INTO tickets VALUES ('London', '$799')")
cursor.execute("INSERT INTO tickets VALUES ('Paris', '$899')")
cursor.execute("INSERT INTO tickets VALUES ('Tokyo', '$1400')")
cursor.execute("INSERT INTO tickets VALUES ('Sydney', '$1200')")
conn.commit()
print("Database created and populated successfully!")

# 3. UPGRADE OUR TOOL to query the database instead of a dictionary!
def get_ticket_price(destination_city):
    print(f"🗄️ [DATABASE] Running SQL query for: {destination_city}...")
    
    # Run a safe parameterized SQL query
    cursor.execute("SELECT price FROM tickets WHERE city=?", (destination_city,))
    result = cursor.fetchone()
    
    # If a result is found, return the price, otherwise say not found
    if result:
        return result[0]
    else:
        return "Price not found in database."

# 4. Launch the Final Phase 4 UI 
gr.ChatInterface(fn=chat,title="FlightAI Assistant").launch()

Database created and populated successfully!
* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
